# Robust Kaggle Smoke Test for OpenVLA (LIBERO-Object)
This version incorporates comprehensive feedback: it handles dependency installation securely before importing any packages, dynamically patches out unnecessary DROID dependencies (like tensorflow_graphics) rather than faking them, ensures strict version preflights, uses robust pathing, lowers the batch size for the T4, and verifies the generated checkpoint thoroughly.

### 1. Safely Install Pinned Dependencies
We run `pip install` via a subprocess with `check=True` to ensure the notebook immediately halts if any installation fails. We do this *before* importing PyTorch to avoid kernel module mismatch.

In [ ]:
import subprocess
import sys

def install(packages, no_deps=False):
    cmd = [sys.executable, "-m", "pip", "install"]
    if no_deps:
        cmd.append("--no-deps")
    cmd.extend(packages.split())
    print(f"Running: {' '.join(cmd)}")
    subprocess.run(cmd, check=True)

# OpenVLA Core Pinned Stack
install("torch==2.2.0 torchvision==0.17.0 torchaudio==2.2.0 transformers==4.40.1 tokenizers==0.19.1 timm==0.9.10 peft==0.11.1 sentencepiece==0.1.99 draccus==0.8.0")

# Standard Utilities
install("einops jsonlines rich matplotlib huggingface_hub bitsandbytes==0.43.1 wandb")

# TensorFlow Datasets
install("tensorflow_datasets==4.9.3")

# DLIMP Dataloader
install("git+https://github.com/moojink/dlimp_openvla", no_deps=True)

print("\n✅ All installations completed successfully!")
print("⚠️ IMPORTANT: If you are running this notebook for the first time, RESTART THE KERNEL now to ensure the new PyTorch version is loaded.")

### 2. Configure Global Paths and Clone OpenVLA

In [ ]:
import os
import subprocess

WORKDIR = "/kaggle/working"
OPENVLA_DIR = os.path.join(WORKDIR, "openvla")
DATA_DIR = os.path.join(WORKDIR, "datasets", "openvla", "modified_libero_rlds")
CHECKPOINT_DIR = os.path.join(WORKDIR, "openvla_checkpoints")

if not os.path.exists(OPENVLA_DIR):
    print("Cloning OpenVLA...")
    subprocess.run(["git", "clone", "https://github.com/openvla/openvla", OPENVLA_DIR], check=True)
else:
    print("✅ OpenVLA already cloned.")

# Ensure OpenVLA modules can be imported
os.environ['PYTHONPATH'] = f"{OPENVLA_DIR}:{os.environ.get('PYTHONPATH', '')}"

### 3. Patch Unnecessary DROID Dependencies (`tensorflow_graphics`)
Since we are training on LIBERO, we don't need the DROID transformation utilities. We patch `droid_utils.py` to remove the `tensorflow_graphics` import, which completely sidesteps the TF Addons compatibility nightmare on Python 3.12.

In [ ]:
patch_file = os.path.join(OPENVLA_DIR, "prismatic", "vla", "datasets", "rlds", "oxe", "utils", "droid_utils.py")
with open(patch_file, "r") as f:
    content = f.read()

if "import tensorflow_graphics.geometry.transformation as tfg" in content:
    content = content.replace("import tensorflow_graphics.geometry.transformation as tfg", 
                              "# import tensorflow_graphics.geometry.transformation as tfg")
    with open(patch_file, "w") as f:
        f.write(content)
    print("✅ Patched droid_utils.py to remove tensorflow_graphics dependency.")
else:
    print("✅ droid_utils.py already patched.")

### 4. Preflight Version Audit & Import Check
Verifies that the versions actually loaded into memory are the ones we explicitly pinned.

In [ ]:
import sys
import torch
import transformers
import peft
import timm
import tensorflow as tf
import tensorflow_datasets as tfds
import bitsandbytes as bnb
import dlimp

print("--- VERSION AUDIT ---")
print(f"Python:       {sys.version.split()[0]}")
print(f"PyTorch:      {torch.__version__} (CUDA: {torch.version.cuda})")
print(f"Transformers: {transformers.__version__}")
print(f"PEFT:         {peft.__version__}")
print(f"timm:         {timm.__version__}")
print(f"BitsAndBytes: {bnb.__version__}")
print(f"TensorFlow:   {tf.__version__}")
print(f"TFDS:         {tfds.__version__}")

print("\n--- HARDWARE AUDIT ---")
if torch.cuda.is_available():
    gpu_count = torch.cuda.device_count()
    print(f"GPUs Detected: {gpu_count}")
    for i in range(gpu_count):
        print(f" - GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GB)")
else:
    raise RuntimeError("❌ No GPU found! Please enable a GPU accelerator in Kaggle.")

### 5. Download the LIBERO-Object RLDS Dataset

In [ ]:
os.makedirs(DATA_DIR, exist_ok=True)
print("Downloading LIBERO dataset...")
subprocess.run([
    "huggingface-cli", "download", "openvla/modified_libero_rlds",
    "--repo-type", "dataset",
    "--include", "libero_object_no_noops/*",
    "--local-dir", DATA_DIR
], check=True)

if os.path.exists(os.path.join(DATA_DIR, "libero_object_no_noops")):
    print("✅ Dataset verified!")
else:
    raise RuntimeError("❌ Dataset download failed or path is incorrect!")

### 6. Run the OpenVLA Smoke Test (Batch=1, Quantized)
Uses `batch_size=1` and `use_quantization=True` to guarantee fitting on a 16GB T4. Runs exactly 10 steps to verify end-to-end functionality.

In [ ]:
os.environ["WANDB_MODE"] = "disabled"

finetune_script = os.path.join(OPENVLA_DIR, "vla-scripts", "finetune.py")
adapter_tmp_dir = os.path.join(CHECKPOINT_DIR, "tmp")

cmd = [
    "torchrun", "--standalone", "--nnodes=1", "--nproc-per-node=1", 
    finetune_script,
    "--vla_path", "openvla/openvla-7b",
    "--data_root_dir", DATA_DIR,
    "--dataset_name", "libero_object_no_noops",
    "--run_root_dir", CHECKPOINT_DIR,
    "--adapter_tmp_dir", adapter_tmp_dir,
    "--use_lora", "True",
    "--use_quantization", "True",
    "--lora_rank", "32",
    "--batch_size", "1",
    "--grad_accumulation_steps", "1",
    "--learning_rate", "5e-4",
    "--image_aug", "False",
    "--wandb_project", "",
    "--wandb_entity", "",
    "--save_steps", "10",
    "--max_steps", "10"
]

print(f"Running command:\n{' '.join(cmd)}\n")
subprocess.run(cmd, check=True)
print("\n✅ Training process exited successfully!")

### 7. Genuine End-to-End Checkpoint Verification
We don't just check if files exist; we verify both the adapter weights and the config.

In [ ]:
import glob
import json

adapters = glob.glob(os.path.join(CHECKPOINT_DIR, "**", "adapter_model.safetensors"), recursive=True)
configs = glob.glob(os.path.join(CHECKPOINT_DIR, "**", "adapter_config.json"), recursive=True)

if adapters and configs:
    adapter_path = adapters[0]
    config_path = configs[0]
    print(f"✅ SUCCESS! Found LoRA adapter at: {adapter_path}")
    
    size_mb = os.path.getsize(adapter_path) / (1024 * 1024)
    print(f"Adapter Size: {size_mb:.2f} MB")
    
    with open(config_path, "r") as f:
        cfg = json.load(f)
        print(f"LoRA Rank (r): {cfg.get('r')}")
        print(f"Target Modules: {cfg.get('target_modules')}")
else:
    raise RuntimeError("❌ Failed to find adapter_model.safetensors or adapter_config.json. The checkpoint was not saved correctly.")